In [1]:
import pandas as pd
import numpy as np

Define Variables

In [2]:
INITIAL_CASH = 100000
LOT_SIZE = 100
DATA_PATH = "interday (intel-assignment).csv"
ANNUAL_FEE_RATE = 0.0005  # 0.05%
DAILY_FEE_RATE = ANNUAL_FEE_RATE / 365
start_date = pd.to_datetime("2018-12-31")
end_date = pd.to_datetime("2021-12-31")
MAX_HOLDING_STOCKS_NUMBER = 4

In [3]:
rank_schedule = {
    "2018-12-31": [2573042, 2572286, 1232815, 2572066],
    "2019-03-31": [2573125, 3695, 2572066, 2572067],
    "2019-06-30": [3695, 8893, 2572065, 2572067],
    "2019-09-30": [2573062, 2572286, 3695, 2572067],
    "2019-12-31": [2572856, 2572065, 851607, 2572765],
    "2020-03-31": [2573062, 851607, 497280, 2572066],
    "2020-06-30": [851607, 2572066, 497280, 2572067],
    "2020-09-30": [4572, 851607, 2572066, 1232815],
    "2020-12-31": [4572, 2572067, 2572765, 3695],
    "2021-03-31": [1232815, 2572067, 2572065, 7634],
    "2021-06-30": [2573085, 2572065, 2573062, 497280],
    "2021-09-30": [7634, 497280, 3695, 1232815],
}


Create Intraday Stock Price Dataframe

In [4]:
df = pd.read_csv(DATA_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"])
prices = df.pivot_table(index="timestamp", columns="jitta_stock_id", values="close")

In [5]:
prices

jitta_stock_id,3695,4572,7634,8893,497280,851607,1232815,2572065,2572066,2572067,2572286,2572765,2572856,2573042,2573062,2573085,2573125,2577965,2583162
timestamp,,,,,,,,,,,,,,,,,,,
2018-12-31,20.952306,7.998225,15.845512,32.126911,NaN,NaN,84.939420,54.367336,29.857637,19.442551,39.177387,32.883053,NaN,16.444390,15.056340,26.239624,22.384640,NaN,NaN
2019-01-02,20.999730,8.056324,15.850127,32.155889,NaN,NaN,85.562258,53.733930,30.173486,19.548584,39.207240,32.697659,NaN,16.335140,14.849654,26.336880,23.712999,NaN,NaN
2019-01-03,20.202993,7.988542,15.236825,31.305869,NaN,NaN,80.423851,52.975763,29.490857,18.862455,37.734481,31.819477,NaN,15.927514,14.331890,25.539382,23.730950,NaN,NaN
2019-01-04,21.170460,8.230619,15.825877,32.378053,NaN,NaN,84.170606,54.501695,31.061813,19.693174,39.615234,32.902568,NaN,16.584708,14.900278,26.511940,24.314351,NaN,NaN
2019-01-07,21.483464,8.298401,16.179309,32.870679,NaN,NaN,85.795822,54.655248,32.078453,20.184780,40.381467,33.624629,NaN,16.584708,14.811947,26.988494,24.987506,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-12-27,40.908689,21.197521,39.522812,66.770000,53.23,28.310355,316.560000,90.300000,57.290000,42.150000,61.440000,62.112700,44.57,45.057775,28.183962,83.433938,11.536400,24.55,15.60
2021-12-28,40.121436,20.987645,39.263581,66.390000,52.32,28.220985,313.440000,90.100000,56.630000,41.610000,60.680000,61.760000,43.99,44.390621,28.376463,84.272470,11.280000,24.53,15.37
2021-12-29,39.836887,20.917686,39.313433,66.350000,52.05,27.992596,313.390000,90.540000,55.920000,41.290900,59.490000,61.700000,43.97,44.050000,27.830000,84.312400,11.050000,24.32,15.25


In [6]:
rank_ids = sorted({sid for stocks in rank_schedule.values() for sid in stocks})

In [7]:
rank_ids = set(rank_ids)  # จากเซลล์ก่อนหน้า
excluded_ids = sorted(set(prices.columns) - rank_ids)

excluded_ids[:20], len(excluded_ids)


([2577965, 2583162], 2)

In [8]:
rank_ids = sorted({sid for stocks in rank_schedule.values() for sid in stocks})
df_prices = prices.reindex(columns=rank_ids)
trading_dates = df_prices.index 
def map_to_trading_date(raw_date):
    if raw_date in trading_dates:
        return raw_date
    pos = trading_dates.searchsorted(raw_date)
    if pos >= len(trading_dates):
        raise ValueError(f"No trading date on or after {raw_date}")
    return trading_dates[pos]
effective_schedule = {
    map_to_trading_date(pd.to_datetime(d)): stocks
    for d, stocks in rank_schedule.items()
}

In [9]:
effective_schedule

{Timestamp('2018-12-31 00:00:00'): [2573042, 2572286, 1232815, 2572066],
 Timestamp('2019-04-01 00:00:00'): [2573125, 3695, 2572066, 2572067],
 Timestamp('2019-07-01 00:00:00'): [3695, 8893, 2572065, 2572067],
 Timestamp('2019-09-30 00:00:00'): [2573062, 2572286, 3695, 2572067],
 Timestamp('2019-12-31 00:00:00'): [2572856, 2572065, 851607, 2572765],
 Timestamp('2020-03-31 00:00:00'): [2573062, 851607, 497280, 2572066],
 Timestamp('2020-06-30 00:00:00'): [851607, 2572066, 497280, 2572067],
 Timestamp('2020-09-30 00:00:00'): [4572, 851607, 2572066, 1232815],
 Timestamp('2020-12-31 00:00:00'): [4572, 2572067, 2572765, 3695],
 Timestamp('2021-03-31 00:00:00'): [1232815, 2572067, 2572065, 7634],
 Timestamp('2021-06-30 00:00:00'): [2573085, 2572065, 2573062, 497280],
 Timestamp('2021-09-30 00:00:00'): [7634, 497280, 3695, 1232815]}

In [10]:
def rebalance(date, target_stocks, holdings, cash):
    total_value = cash
    price_row = df_prices.loc[date]  
    for sid, shares in holdings.items():
        if shares == 0:
            continue
        px = price_row[sid]
        if pd.isna(px):
            raise ValueError(f"Missing price for holding {sid} on {date}")
        total_value += shares * px
    # เป้าหมายต่อหุ้น (เท่ากันเท่าที่ทำได้)
    target_value = total_value / len(target_stocks)

    # จำนวนหุ้นที่ควรถือ (ปัดลงตาม lot)
    desired = {}
    for sid in target_stocks:
        px = price_row[sid]
        lots = np.floor(target_value / (px * LOT_SIZE))
        desired[sid] = int(lots * LOT_SIZE)

    # ขายก่อน (หุ้นที่มีมากกว่าเป้าหมาย + หุ้นที่หลุด rank)
    all_sids = sorted(set(holdings.keys()) | set(target_stocks))
    for sid in all_sids:
        cur = holdings.get(sid, 0)
        tgt = desired.get(sid, 0)
        if cur > tgt:
            shares = cur - tgt
            cash += shares * price_row[sid]
            holdings[sid] = tgt
            
    # ซื้อทีหลัง (ถ้าเงินไม่พอ ให้ซื้อได้เท่าที่ทำได้)
    for sid in target_stocks:
        cur = holdings.get(sid, 0)
        tgt = desired.get(sid, 0)
        if cur < tgt:
            shares = tgt - cur
            cost = shares * price_row[sid]
            if cost > cash + 1e-9:
                max_lots = int(cash // (price_row[sid] * LOT_SIZE))
                shares = max_lots * LOT_SIZE
                if shares == 0:
                    continue
                cost = shares * price_row[sid]
                tgt = cur + shares
            cash -= cost
            holdings[sid] = tgt
    return holdings, cash

In [11]:
stock_cols = [f"stock{i}" for i in range(1, MAX_HOLDING_STOCKS_NUMBER + 1)]
lot_cols = [f"lot{i}" for i in range(1, MAX_HOLDING_STOCKS_NUMBER + 1)]
value_cols = [f"value{i}" for i in range(1, MAX_HOLDING_STOCKS_NUMBER + 1)]

holdings = {}
cash = INITIAL_CASH
accrued_fee = 0.0
current_order = []

nav_rows = []
rank_schedule_dt = {pd.to_datetime(k): v for k, v in rank_schedule.items()}
rebalance_dates = pd.to_datetime(list(rank_schedule.keys()))
rebalance_schedule = set(rebalance_dates)


# print(rebalance_schedule)
dates = pd.date_range(start_date, end_date, freq="D")
daily_prices = df_prices.reindex(dates).ffill()
for date in dates:
    is_rebalance = False
    is_payfee = False

    if date in effective_schedule:
        current_order = effective_schedule[date]
        holdings, cash = rebalance(date, current_order, holdings, cash)
        print(current_order)
        is_rebalance = True
       
    held = [sid for sid in current_order if holdings.get(sid, 0) > 0]
    held = held[:MAX_HOLDING_STOCKS_NUMBER]
    held = held + [None] * (MAX_HOLDING_STOCKS_NUMBER - len(held))

    price_row = daily_prices.loc[date]
    stock_value_total = 0.0
    for sid, shares in holdings.items():
        if shares == 0:
            continue
        stock_value_total += shares * price_row[sid]

    asset_value = stock_value_total + cash - accrued_fee
    daily_fee = asset_value * DAILY_FEE_RATE
    accrued_fee += daily_fee

    if date.month == 12 and date.day == 31 and accrued_fee > 0:
        cash -= accrued_fee
        accrued_fee = 0.0
        is_payfee = True
    lots = []
    values = []
    for sid in held:
        if sid is None:
            lots.append(0)
            values.append(0.0)
        else:
            shares = holdings[sid]
            lots.append(shares // LOT_SIZE)
            values.append(shares * price_row[sid])
    nav = stock_value_total + cash - accrued_fee
    nav_rows.append({
        "date": date,
        **{stock_cols[i]: held[i] for i in range(MAX_HOLDING_STOCKS_NUMBER)},
        **{lot_cols[i]: lots[i] for i in range(MAX_HOLDING_STOCKS_NUMBER)},
        **{value_cols[i]: values[i] for i in range(MAX_HOLDING_STOCKS_NUMBER)},
        "stock_value_total": sum(values),
        "cash": cash,
        "accrued_fee": accrued_fee,
        "nav": nav,
        "is_rebalance": is_rebalance,
        "is_payfee": is_payfee,
    })

df_nav = pd.DataFrame(nav_rows).set_index("date").sort_index()


[2573042, 2572286, 1232815, 2572066]
[2573125, 3695, 2572066, 2572067]
[3695, 8893, 2572065, 2572067]
[2573062, 2572286, 3695, 2572067]
[2572856, 2572065, 851607, 2572765]
[2573062, 851607, 497280, 2572066]
[851607, 2572066, 497280, 2572067]
[4572, 851607, 2572066, 1232815]
[4572, 2572067, 2572765, 3695]
[1232815, 2572067, 2572065, 7634]
[2573085, 2572065, 2573062, 497280]
[7634, 497280, 3695, 1232815]


In [13]:
df_nav.to_csv("nav.csv", index=True)